In [1]:
from graphdatascience import GraphDataScience
from neo4j import GraphDatabase

graphdb = GraphDatabase.driver(
            "bolt://localhost:7687",
            auth=("neo4j", "123456789")
        )

gds = GraphDataScience(
            "bolt://localhost:7687",
            auth=("neo4j", "123456789")
        )

d:\HocTap\ChatBot\Medical_Chatbot\develop-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
            model="qwen-3-1.7b",
            temperature=0.2,
            top_p=0.5,
            api_key="abcd",  # Sử dụng khóa API đầu tiên từ danh sách
            base_url="http://localhost:8000/v1",
            extra_body={
                "chat_template_kwargs": {
                    "enable_thinking": False
                }
            },
        )

In [3]:
print("[Community] Removing old communities...")

result = graphdb.execute_query(
    """
    MATCH (c:Community)
    DETACH DELETE c
    RETURN count(c) AS deleted
    """
)

deleted = result.records[0]["deleted"]

print(
    f"[Community] "
    f"Deleted {deleted} old communities"
)

[Community] Removing old communities...
[Community] Deleted 211 old communities


In [4]:
# =========================================================
# 2. PROJECT GRAPH FOR LEIDEN
# =========================================================

if gds.graph.exists("entity_graph")["exists"]:
    gds.graph.drop("entity_graph")

G, project_result = gds.graph.project(
    "entity_graph",
    "EntityMention",
    {
        "RELATED_TO": {
            "orientation": "UNDIRECTED",
            "properties": "weight"
        }
    },
)

print(
    f"[Community] "
    f"Graph projected: {project_result}"
)

[Community] Graph projected: nodeProjection            {'EntityMention': {'properties': {}, 'label': ...
relationshipProjection    {'RELATED_TO': {'orientation': 'UNDIRECTED', '...
graphName                                                      entity_graph
nodeCount                                                               339
relationshipCount                                                       524
projectMillis                                                            34
Name: 0, dtype: object


In [ ]:
# =========================================================
# 3. RUN LEIDEN
# =========================================================

leiden_result = gds.leiden.write(
    G,
    relationshipWeightProperty="weight",
    writeProperty="communities",
    includeIntermediateCommunities=True,
    maxLevels=3,
)

print(
    "[Community] "
    f"Leiden completed: {leiden_result}"
)

[Community] Leiden completed: ranLevels                                                                2
didConverge                                                           True
nodeCount                                                              339
communityCount                                                          98
preProcessingMillis                                                      1
computeMillis                                                           70
postProcessingMillis                                                     2
writeMillis                                                             13
nodePropertiesWritten                                                  339
communityDistribution    {'p1': 1, 'p5': 1, 'max': 28, 'p90': 8, 'p50':...
modularities                       [0.878748435963397, 0.9203939100693043]
modularity                                                        0.920394
configuration            {'writeConcurrency': 4, 'consecutiveIds': Fal

In [6]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Startup] Using device: {device}")

embedding_model = HuggingFaceEmbeddings(
    model_name= "bkai-foundation-models/vietnamese-bi-encoder",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": False},
)

[Startup] Using device: cuda


In [7]:
# def create_communities(self):
"""
Tạo lại toàn bộ Community từ kết quả Leiden.

EntityMention:
    communities = [15, 20]

Sẽ tạo:
    Community 0-15
    Community 1-20

Và:
    EntityMention -[:IN_COMMUNITY]-> Community
"""

print("[Community] Creating communities...")

result = graphdb.execute_query(
    """
    MATCH (e:EntityMention)

    WHERE e.communities IS NOT NULL
        AND size(e.communities) > 0

    UNWIND range(0, size(e.communities) - 1) AS level

    WITH
        e,
        level,
        e.communities[level] AS leiden_id

    MERGE (c:Community {
        community_id:
            toString(level) + "-" + toString(leiden_id)
    })

    SET
        c.level = level,
        c.leiden_community_id = leiden_id,
        c.is_active = true,
        c.updated_at = datetime()

    MERGE (e)-[:IN_COMMUNITY]->(c)

    RETURN count(*) AS entity_community_links
    """
)

count = result.records[0]["entity_community_links"]

print(
    f"[Community] "
    f"Created/updated {count} entity-community links"
)

[Community] Creating communities...
[Community] Created/updated 678 entity-community links


In [8]:
# def create_community_hierarchy(self):
"""
Tạo quan hệ phân cấp giữa các Community.

communities = [15, 20, 35]

Sẽ tạo:

    Community 0-15
            |
            v
    Community 1-20
            |
            v
    Community 2-35
"""

print("[Community] Creating community hierarchy...")

result = graphdb.execute_query(
    """
    MATCH (e:EntityMention)

    WHERE e.communities IS NOT NULL
        AND size(e.communities) > 1

    UNWIND range(1, size(e.communities) - 1) AS level

    WITH
        level,
        e.communities[level - 1] AS parent_leiden_id,
        e.communities[level] AS child_leiden_id

    MATCH (parent:Community {
        community_id:
            toString(level - 1)
            + "-"
            + toString(parent_leiden_id)
    })

    MATCH (child:Community {
        community_id:
            toString(level)
            + "-"
            + toString(child_leiden_id)
    })

    MERGE (parent)-[:PARENT_OF]->(child)

    RETURN count(*) AS hierarchy_links
    """
)

count = result.records[0]["hierarchy_links"]

print(
    f"[Community] "
    f"Created {count} community hierarchy links"
)

[Community] Creating community hierarchy...
[Community] Created 339 community hierarchy links


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed


# def generate_all_community_reports(
#     self,
#     max_workers: int = 8,
# ):
"""
Generate / update report cho toàn bộ Community.

Pipeline:

    Community
        ↓
    Load EntityMention
        ↓
    Load RELATED_TO
        ↓
    Build context
        ↓
    LLM Generate Report
        ↓
    Generate Summary Embedding
        ↓
    Save Community
"""

# =========================================================
# 1. LOAD ALL COMMUNITIES
# =========================================================

print("[Community] Loading communities...")

communities_result = graphdb.execute_query(
    """
    MATCH (c:Community)

    RETURN
        c.community_id AS community_id,
        c.level AS level,
        c.leiden_community_id AS leiden_community_id
    ORDER BY
        c.level,
        c.community_id
    """
)

communities = []

for record in communities_result.records:

    communities.append(
        {
            "community_id": record["community_id"],
            "level": record["level"],
            "leiden_community_id": (
                record["leiden_community_id"]
            ),
        }
    )

print(
    f"[Community] "
    f"Found {len(communities)} communities"
)


[Community] Loading communities...
[Community] Found 211 communities


In [11]:
from pydantic import BaseModel, Field


class CommunityReport(BaseModel):
    title: str = Field(
        description="Short title describing the main topic of the community."
    )

    summary: str = Field(
        description="Concise factual summary of the community."
    )

    key_entities: list[str] = Field(
        description="Important entities in the community."
    )

    key_relationships: list[str] = Field(
        description="Important relationships between entities."
    )

    findings: list[str] = Field(
        description="Important factual findings derived from the community."
    )

In [13]:
import json


def generate_community_report(
    community_context: dict,
) -> CommunityReport | None:

    try:

        prompt = f"""
You are an expert in medical knowledge graph analysis.

Analyze the following community of medical entities
and relationships.

Generate a concise and factual community report.

Community:
{json.dumps(
    community_context,
    ensure_ascii=False,
    indent=2,
)}

Rules:
- Use only information provided in the community.
- Do not invent medical facts.
- Identify the main topic.
- Summarize important entities and relationships.
- Focus on medically meaningful information.
"""

        structured_llm = llm.with_structured_output(
            CommunityReport
        )

        response = structured_llm.invoke(
            [
                {
                    "role": "system",
                    "content": (
                        "You are an expert in "
                        "medical knowledge graph analysis."
                    ),
                },
                {
                    "role": "user",
                    "content": prompt,
                },
            ]
        )

        return response

    except Exception as e:

        print(
            "[Community Report] "
            f"Failed to generate report: {e}"
        )

        return None

In [14]:


# =========================================================
# 2. PROCESS ONE COMMUNITY
# =========================================================

def process_community(community):

    community_id = community["community_id"]
    level = community["level"]

    print(
        f"[Community] Processing "
        f"{community_id} "
        f"(level={level})"
    )

    # =====================================================
    # 2.1 LOAD ENTITIES
    # =====================================================

    entities_result = graphdb.execute_query(
        """
        MATCH (e:EntityMention)
            -[:IN_COMMUNITY]->
            (c:Community)

        WHERE c.community_id = $community_id

        RETURN
            e.mention_id AS mention_id,
            e.name AS name,
            e.type AS type,
            e.description AS description
        """,
        community_id=community_id,
    )

    entity_context = []

    for record in entities_result.records:

        entity_context.append(
            {
                "mention_id": record["mention_id"],
                "name": record["name"],
                "type": record["type"],
                "description": (
                    record["description"]
                    or ""
                ),
            }
        )

    if not entity_context:

        print(
            f"[Community] "
            f"{community_id} has no entities"
        )

        return None

    # =====================================================
    # 2.2 LOAD RELATIONSHIPS
    # =====================================================

    relationships_result = graphdb.execute_query(
        """
        MATCH (a:EntityMention)
            -[r:RELATED_TO]->
            (b:EntityMention)

        MATCH (a)-[:IN_COMMUNITY]->(c:Community)

        MATCH (b)-[:IN_COMMUNITY]->(c)

        WHERE c.community_id = $community_id

        RETURN
            a.name AS source,
            b.name AS target,
            r.description AS description,
            r.weight AS weight
        """,
        community_id=community_id,
    )

    relationship_context = []

    for record in relationships_result.records:

        relationship_context.append(
            {
                "source": record["source"],
                "target": record["target"],
                "description": (
                    record["description"]
                    or ""
                ),
                "weight": (
                    record["weight"]
                    or 1.0
                ),
            }
        )

    # =====================================================
    # 2.3 BUILD COMMUNITY CONTEXT
    # =====================================================

    community_context = {
        "community_id": community_id,
        "level": level,
        "entities": entity_context,
        "relationships": relationship_context,
    }

    print(
        f"[Community] {community_id}: "
        f"{len(entity_context)} entities, "
        f"{len(relationship_context)} relationships"
    )

    # =====================================================
    # 2.4 GENERATE REPORT
    # =====================================================

    # Nếu chỉ có 1 entity thì không cần gọi LLM
    if len(entity_context) == 1:

        entity = entity_context[0]

        report = CommunityReport(
            title=entity["name"],
            summary=(
                entity["description"]
                or entity["name"]
            ),
            key_entities=[
                entity["name"]
            ],
            key_relationships=[],
            findings=[
                entity["description"]
            ] if entity["description"] else [],
        )

        print(
            f"[Community] "
            f"{community_id}: "
            f"single entity, skip LLM"
        )

    else:

        report = generate_community_report(
            community_context
        )

        if not report:

            print(
                f"[Community] "
                f"Failed to generate report: "
                f"{community_id}"
            )

            return None

    # =====================================================
    # 2.5 GENERATE EMBEDDING
    # =====================================================

    summary_embedding = (
        embedding_model.embed_query(
            report.summary
        )
    )

    # =====================================================
    # 2.6 SAVE COMMUNITY REPORT
    # =====================================================

    graphdb.execute_query(
        """
        MATCH (c:Community {
            community_id: $community_id
        })

        SET
            c.level = $level,
            c.title = $title,
            c.summary = $summary,
            c.key_entities = $key_entities,
            c.key_relationships = $key_relationships,
            c.findings = $findings,
            c.embedding = $embedding,
            c.is_active = true,
            c.updated_at = datetime()

        RETURN
            c.community_id AS community_id
        """,
        community_id=community_id,
        level=level,
        title=report.title,
        summary=report.summary,
        key_entities=report.key_entities,
        key_relationships=(
            report.key_relationships
        ),
        findings=report.findings,
        embedding=summary_embedding,
    )

    return {
        "community_id": community_id,
        "level": level,
        "entity_count": len(
            entity_context
        ),
        "relationship_count": len(
            relationship_context
        ),
    }



In [15]:
# =========================================================
# 3. RUN IN PARALLEL
# =========================================================

updated_reports = []

with ThreadPoolExecutor(
    max_workers=8
) as executor:

    futures = {
        executor.submit(
            process_community,
            community,
        ): community["community_id"]
        for community in communities
    }

    total = len(futures)

    for completed, future in enumerate(
        as_completed(futures),
        start=1,
    ):

        community_id = futures[future]

        try:

            result = future.result()

            if result:
                updated_reports.append(
                    result
                )

        except Exception as e:

            print(
                f"[Community] "
                f"Community {community_id} "
                f"failed: {e}"
            )

        print(
            f"[Community] "
            f"Progress: "
            f"{completed}/{total}"
        )

print(
    f"[Community] "
    f"Updated {len(updated_reports)} "
    f"community reports"
)

# return updated_reports

[Community] Processing 0-1 (level=0)
[Community] Processing 0-10 (level=0)
[Community] Processing 0-102 (level=0)
[Community] Processing 0-103 (level=0)
[Community] Processing 0-107 (level=0)
[Community] Processing 0-112 (level=0)
[Community] Processing 0-118 (level=0)
[Community] Processing 0-123 (level=0)
[Community] 0-103: 1 entities, 0 relationships
[Community] 0-103: single entity, skip LLM
[Community] 0-112: 1 entities, 0 relationships
[Community] 0-112: single entity, skip LLM
[Community] 0-10: 2 entities, 1 relationships
[Community] 0-123: 5 entities, 4 relationships
[Community] 0-102: 4 entities, 3 relationships
[Community] 0-107: 9 entities, 8 relationships
[Community] 0-118: 10 entities, 9 relationships
[Community] 0-1: 14 entities, 14 relationships
[Community] Processing 0-126 (level=0)
[Community] Progress: 1/211
[Community] Processing 0-127 (level=0)
[Community] Progress: 2/211
[Community] 0-126: 1 entities, 0 relationships
[Community] 0-126: single entity, skip LLM
[Comm